# 01 - Data Merging: PISA 2022 Process Data Analysis

## Purpose
This notebook merges three PISA 2022 international data files into a single 
analysis-ready dataset for downstream machine learning modeling.

## Data Sources
- **CY08MSP_STU_COG.SAV** (3.5 GB): Cognitive assessment with process data 
  (mathematics CM/DM columns) and 10 plausible values (PV1MATH - PV10MATH)
- **CY08MSP_STU_QQQ.SAV** (2 GB): Student background questionnaire 
  (ESCS, gender, parental education, self-efficacy, anxiety, etc.)
- **CY08MSP_STU_TIM.SAV** (633 MB): Timing data (only EFFORTT retained as 
  effort/motivation indicator)

## Output
- `data/analysis/pisa2022_ml_dataset.parquet`: Merged dataset with student-level 
  mathematics achievement, process features, and contextual variables.

## Strategy
Given the size of COG.SAV (3.5 GB), we use chunked reading via pyreadstat to 
avoid memory overflow on Apple Silicon. Non-mathematics process columns and 
questionnaire timing columns are excluded early to reduce memory footprint.

In [1]:
# Imports and environment check

# Standard libraries for data handling, file I/O, and memory monitoring.
# pyreadstat is used for reading SPSS .SAV files with metadata preservation.

import os
import gc  # Garbage collector for manual memory release after large reads
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat  # SPSS .SAV reader with chunked reading support
import psutil  # For monitoring memory usage during large file operations

warnings.filterwarnings("ignore")

# Display settings for pandas: show more columns and wider output
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Environment check: confirm Python version and key package versions
print("Environment check")
print("-" * 50)
print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print(f"pyreadstat   : {pyreadstat.__version__}")
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.2f} GB")
print(f"Total RAM    : {psutil.virtual_memory().total / 1e9:.2f} GB")

Environment check
--------------------------------------------------
pandas       : 3.0.2
numpy        : 2.4.4
pyreadstat   : 1.3.4
Available RAM: 5.36 GB
Total RAM    : 17.18 GB


In [2]:
# Define project paths

# Using pathlib for cross-platform path handling. Paths are defined relative
# to the project root so the notebook remains portable.

# Project root: two levels up from notebooks/process_data/
PROJECT_ROOT = Path.home() / "Desktop" / "PISA-Process-Data-Analysis"

# Input directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR = PROJECT_ROOT / "data" / "analysis"

# Ensure output directories exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# Input file paths
COG_PATH = RAW_DIR / "CY08MSP_STU_COG.SAV"
QQQ_PATH = RAW_DIR / "CY08MSP_STU_QQQ.SAV"
TIM_PATH = RAW_DIR / "CY08MSP_STU_TIM.SAV"

# Output path
OUTPUT_PATH = ANALYSIS_DIR / "pisa2022_ml_dataset.parquet"

# Verify input files exist and report their sizes
print("Input file check")
print("-" * 60)
for name, path in [("COG", COG_PATH), ("QQQ", QQQ_PATH), ("TIM", TIM_PATH)]:
    if path.exists():
        size_gb = path.stat().st_size / 1e9
        print(f"[OK]      {name}: {path.name}  ({size_gb:.2f} GB)")
    else:
        print(f"[MISSING] {name}: {path}")

print(f"\nOutput will be saved to: {OUTPUT_PATH}")

Input file check
------------------------------------------------------------
[OK]      COG: CY08MSP_STU_COG.SAV  (3.73 GB)
[OK]      QQQ: CY08MSP_STU_QQQ.SAV  (2.10 GB)
[OK]      TIM: CY08MSP_STU_TIM.SAV  (0.66 GB)

Output will be saved to: /Users/mrved/Desktop/PISA-Process-Data-Analysis/data/analysis/pisa2022_ml_dataset.parquet


In [3]:
# Read COG metadata only (no data) to inspect column structure

# pyreadstat's metadataonly=True reads only the file header without loading
# the actual data. This is critical for a 3.73 GB file: we first identify
# which columns we need, then read only those columns in the next step.
# This approach reduces memory usage dramatically.

print("Reading COG metadata (header only, no data)...")
start = time.time()

# metadataonly=True returns an empty DataFrame but a full metadata object
_, cog_meta = pyreadstat.read_sav(
    str(COG_PATH),
    metadataonly=True
)

elapsed = time.time() - start
print(f"Metadata read in {elapsed:.1f} seconds")
print(f"Total columns in COG: {len(cog_meta.column_names)}")
print(f"Total rows in COG   : {cog_meta.number_rows:,}")

Reading COG metadata (header only, no data)...
Metadata read in 0.2 seconds
Total columns in COG: 5023
Total rows in COG   : 613,744


In [4]:
# Categorize COG columns and identify mathematics process columns

# PISA 2022 CM columns contain process data distinguished by suffix:
#   TT : Total time on item
#   A  : Number of actions
#   V  : Number of visits
#   F  : First-visit duration
#   VS : Visit sequence

import re

all_cols = cog_meta.column_names

# Helper function to extract trailing alphabetic suffix
def get_suffix(col):
    match = re.search(r'[A-Z]+$', col)
    return match.group() if match else ""

# ID columns for joining with QQQ and TIM
id_cols = [c for c in all_cols if c in ("CNTSTUID", "CNT", "CNTSCHID")]

# Process data suffixes to keep
process_suffixes = {"TT", "A", "V", "F", "VS"}

# Extract CM columns with process-data suffixes
cm_process_cols = [
    c for c in all_cols 
    if c.startswith("CM") and get_suffix(c) in process_suffixes
]

# Final COG column list
cog_cols_to_read = id_cols + cm_process_cols

print("COG column selection summary:")
print(f"  ID columns: {len(id_cols)} -> {id_cols}")
print(f"  CM process columns: {len(cm_process_cols)}")
print(f"  Total columns to read: {len(cog_cols_to_read)}")
print(f"  Reduction: {len(cog_cols_to_read)}/{len(all_cols)} = {100*len(cog_cols_to_read)/len(all_cols):.1f}%")

print("\nBreakdown by suffix:")
for suf in sorted(process_suffixes):
    count = sum(1 for c in cm_process_cols if get_suffix(c) == suf)
    print(f"  {suf}: {count}")

print("\nExample columns per suffix:")
for suf in sorted(process_suffixes):
    examples = [c for c in cm_process_cols if get_suffix(c) == suf][:3]
    print(f"  {suf}: {examples}")

COG column selection summary:
  ID columns: 3 -> ['CNT', 'CNTSCHID', 'CNTSTUID']
  CM process columns: 1170
  Total columns to read: 1173
  Reduction: 1173/5023 = 23.4%

Breakdown by suffix:
  A: 234
  F: 234
  TT: 234
  V: 234
  VS: 234

Example columns per suffix:
  A: ['CM033Q01A', 'CM474Q01A', 'CM155Q02A']
  F: ['CM033Q01F', 'CM474Q01F', 'CM155Q02F']
  TT: ['CM033Q01TT', 'CM474Q01TT', 'CM155Q02TT']
  V: ['CM033Q01V', 'CM474Q01V', 'CM155Q02V']
  VS: ['CM033Q01VS', 'CM474Q01VS', 'CM155Q02VS']


In [5]:
# Define columns to read from QQQ and TIM files
# QQQ: plausible values + contextual variables
# HEDRES not available in PISA 2022; HOMEPOS used instead
# TIM: only EFFORTT retained as effort indicator

qqq_id_cols = ["CNTSTUID"]
qqq_pv_cols = [f"PV{i}MATH" for i in range(1, 11)]
qqq_context_cols = [
    "ESCS",
    "ST004D01T",
    "ST019AQ01T",
    "MISCED",
    "FISCED",
    "HOMEPOS",
    "ICTHOME",
    "MATHEFF",
    "ANXMAT",
    "BELONG",
]

qqq_cols_to_read = qqq_id_cols + qqq_pv_cols + qqq_context_cols
tim_cols_to_read = ["CNTSTUID", "EFFORTT"]

# Verify QQQ columns exist
qqq_available = set(qqq_meta.column_names)
qqq_missing = [c for c in qqq_cols_to_read if c not in qqq_available]

if qqq_missing:
    print(f"WARNING: QQQ columns not found: {qqq_missing}")
else:
    print("All QQQ columns verified present in file.")

print(f"\nQQQ columns to read: {len(qqq_cols_to_read)}")
print(f"  ID: {qqq_id_cols}")
print(f"  Plausible values: {qqq_pv_cols}")
print(f"  Contextual: {qqq_context_cols}")

print(f"\nTIM columns to read: {len(tim_cols_to_read)}")
print(f"  {tim_cols_to_read}")

print("\nOverall read plan:")
print(f"  COG: {len(cog_cols_to_read)} columns")
print(f"  QQQ: {len(qqq_cols_to_read)} columns")
print(f"  TIM: {len(tim_cols_to_read)} columns")

NameError: name 'qqq_meta' is not defined

In [ ]:
# Read COG file in chunks to manage memory for 3.73 GB file

# Strategy:
#   1. Use pyreadstat's read_file_in_chunks generator
#   2. Process 50,000 rows at a time (tunable based on available RAM)
#   3. Keep only the columns we need (cog_cols_to_read)
#   4. Concatenate chunks into a single DataFrame
#   5. Monitor memory after each chunk

# Note: pyreadstat does not support column filtering in chunk mode directly,
# so we filter columns within each chunk before appending.

CHUNK_SIZE = 50_000  # rows per chunk (conservative for 8 GB available RAM)

print(f"Starting COG chunked read")
print(f"  Chunk size: {CHUNK_SIZE:,} rows")
print(f"  Expected chunks: ~{cog_meta.number_rows // CHUNK_SIZE + 1}")
print(f"  Columns to keep: {len(cog_cols_to_read)}")
print("-" * 70)

start_time = time.time()
cog_chunks = []
total_rows_read = 0

# Create chunk reader
chunk_reader = pyreadstat.read_file_in_chunks(
    pyreadstat.read_sav,
    str(COG_PATH),
    chunksize=CHUNK_SIZE,
    usecols=cog_cols_to_read,  # Column filtering at read time
)

# Iterate through chunks
for i, (chunk_df, chunk_meta) in enumerate(chunk_reader, start=1):
    chunk_rows = len(chunk_df)
    total_rows_read += chunk_rows
    
    # Append chunk to list
    cog_chunks.append(chunk_df)
    
    # Report progress every chunk
    elapsed = time.time() - start_time
    mem_avail = psutil.virtual_memory().available / 1e9
    mem_used_pct = psutil.virtual_memory().percent
    
    print(f"Chunk {i:3d} | rows: {chunk_rows:6,} | total: {total_rows_read:7,} "
          f"| elapsed: {elapsed:6.1f}s | RAM avail: {mem_avail:.2f} GB "
          f"| usage: {mem_used_pct:.1f}%")
    
    # Manual garbage collection after each chunk
    gc.collect()

print("-" * 70)
print(f"Chunked read complete. Concatenating {len(cog_chunks)} chunks...")

# Concatenate all chunks into one DataFrame
cog_df = pd.concat(cog_chunks, ignore_index=True)

# Free chunk list memory
del cog_chunks
gc.collect()

elapsed_total = time.time() - start_time
print(f"\nFinal COG DataFrame:")
print(f"  Shape       : {cog_df.shape}")
print(f"  Memory usage: {cog_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"  Total time  : {elapsed_total:.1f} seconds ({elapsed_total/60:.1f} minutes)")
print(f"  RAM available: {psutil.virtual_memory().available / 1e9:.2f} GB")

Starting COG chunked read
  Chunk size: 50,000 rows
  Expected chunks: ~13
  Columns to keep: 1173
----------------------------------------------------------------------
Chunk   1 | rows: 50,000 | total:  50,000 | elapsed:    5.6s | RAM avail: 6.97 GB | usage: 59.4%
Chunk   2 | rows: 50,000 | total: 100,000 | elapsed:   11.1s | RAM avail: 6.34 GB | usage: 63.1%
Chunk   3 | rows: 50,000 | total: 150,000 | elapsed:   17.7s | RAM avail: 6.42 GB | usage: 62.6%
Chunk   4 | rows: 50,000 | total: 200,000 | elapsed:   25.3s | RAM avail: 6.36 GB | usage: 63.0%
Chunk   5 | rows: 50,000 | total: 250,000 | elapsed:   33.9s | RAM avail: 6.38 GB | usage: 62.9%
Chunk   6 | rows: 50,000 | total: 300,000 | elapsed:   43.3s | RAM avail: 6.30 GB | usage: 63.4%
Chunk   7 | rows: 50,000 | total: 350,000 | elapsed:   53.6s | RAM avail: 6.35 GB | usage: 63.1%
Chunk   8 | rows: 50,000 | total: 400,000 | elapsed:   64.9s | RAM avail: 6.25 GB | usage: 63.6%
Chunk   9 | rows: 50,000 | total: 450,000 | elapsed:  

In [ ]:
# Optimize COG DataFrame memory by downcasting numeric types

# pyreadstat reads SPSS numeric columns as float64 (8 bytes). For PISA process
# data, most values are small non-negative integers or counts that fit comfortably
# in smaller types. We downcast to reduce memory footprint roughly by half.

# Strategy:
#   - ID columns (CNTSTUID, CNT, CNTSCHID): keep as-is (critical for joins)
#   - Numeric process columns: downcast to float32 (preserves NaN, halves size)
#   - Integer-like columns would go to int32, but float32 is safer when NaN exists

# Note: float32 has ~7 decimal digits of precision, plenty for count/timing data.

print("Memory before optimization:")
print(f"  Total: {cog_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"  Dtype breakdown:")
print(cog_df.dtypes.value_counts())

# Identify numeric columns excluding IDs
id_col_names = ["CNTSTUID", "CNT", "CNTSCHID"]
numeric_cols = [c for c in cog_df.columns if c not in id_col_names]

# Downcast all numeric process columns to float32
# (float32 preserves NaN and handles the range of PISA values with plenty of headroom)
for col in numeric_cols:
    if pd.api.types.is_numeric_dtype(cog_df[col]):
        cog_df[col] = cog_df[col].astype("float32")

print("\nMemory after optimization:")
print(f"  Total: {cog_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"  Dtype breakdown:")
print(cog_df.dtypes.value_counts())

# Release any temporary memory
gc.collect()

print(f"\nRAM available: {psutil.virtual_memory().available / 1e9:.2f} GB")

Memory before optimization:
  Total: 5.76 GB
  Dtype breakdown:
float64    1172
str           1
Name: count, dtype: int64

Memory after optimization:
  Total: 2.89 GB
  Dtype breakdown:
float32    1170
float64       2
str           1
Name: count, dtype: int64

RAM available: 5.51 GB


In [ ]:
# Quick health check on the COG DataFrame before moving to QQQ
# Verify the data looks reasonable before investing in downstream steps

print("COG DataFrame overview:")
print(f"  Shape: {cog_df.shape}")
print(f"  Unique students (CNTSTUID): {cog_df['CNTSTUID'].nunique():,}")
print(f"  Unique countries (CNT): {cog_df['CNT'].nunique()}")

# Verify no duplicate student IDs in COG (should be 1 row per student)
dup_count = cog_df['CNTSTUID'].duplicated().sum()
print(f"  Duplicate CNTSTUID: {dup_count}")

# Show the first few ID columns and a few process columns
print("\nFirst 5 rows (ID columns + sample process columns):")
sample_process = ["CM033Q01TT", "CM033Q01A", "CM033Q01V", "CM033Q01F", "CM033Q01VS"]
sample_process_available = [c for c in sample_process if c in cog_df.columns]
display_cols = id_col_names + sample_process_available
print(cog_df[display_cols].head())

# Basic statistics on a few process columns to check value ranges
print("\nDescriptive stats on sample TT columns (total time in ms):")
tt_sample = [c for c in cog_df.columns if c.endswith("TT")][:5]
print(cog_df[tt_sample].describe().round(1))

# Check missing data pattern on a few columns
# In PISA, students only see a subset of items, so most process cells ARE NaN by design
print("\nMissing rate on sample process columns (expected: high, by design):")
for col in sample_process_available:
    nan_pct = 100 * cog_df[col].isna().mean()
    print(f"  {col}: {nan_pct:.1f}% missing")

COG DataFrame overview:
  Shape: (613744, 1173)
  Unique students (CNTSTUID): 613,744
  Unique countries (CNT): 80
  Duplicate CNTSTUID: 0

First 5 rows (ID columns + sample process columns):
   CNTSTUID  CNT  CNTSCHID  CM033Q01TT  CM033Q01A  CM033Q01V  CM033Q01F  CM033Q01VS
0  800001.0  ALB  800282.0         NaN        NaN        NaN        NaN         NaN
1  800002.0  ALB  800115.0         NaN        NaN        NaN        NaN         NaN
2  800003.0  ALB  800242.0         NaN        NaN        NaN        NaN         NaN
3  800005.0  ALB  800245.0         NaN        NaN        NaN        NaN         NaN
4  800006.0  ALB  800285.0         NaN        NaN        NaN        NaN         NaN

Descriptive stats on sample TT columns (total time in ms):
       CM033Q01TT  CM474Q01TT  CM155Q02TT  CM155Q01TT  CM155Q04TT
count     96447.0     69048.0     63492.0     63292.0     63158.0
mean      47560.0     56291.3    182556.4     78551.8     88724.6
std       42467.5     51052.6    299946.9     

In [ ]:
# Read QQQ file - contains plausible values and contextual variables

# QQQ is 2 GB total but we only need 21 specific columns, so we use
# usecols parameter to filter at read time. No chunking needed because
# the filtered dataset is small enough to fit comfortably in memory.

print("Reading QQQ file with column filtering...")
start = time.time()

qqq_df, _ = pyreadstat.read_sav(
    str(QQQ_PATH),
    usecols=qqq_cols_to_read,
)

# Downcast numeric columns to float32 to save memory (same rationale as COG)
for col in qqq_df.columns:
    if col == "CNTSTUID":
        continue
    if pd.api.types.is_numeric_dtype(qqq_df[col]):
        qqq_df[col] = qqq_df[col].astype("float32")

elapsed = time.time() - start
print(f"QQQ read in {elapsed:.1f} seconds")
print(f"  Shape: {qqq_df.shape}")
print(f"  Memory: {qqq_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  RAM available: {psutil.virtual_memory().available / 1e9:.2f} GB")

# Quick health check
print("\nQQQ health check:")
print(f"  Unique students: {qqq_df['CNTSTUID'].nunique():,}")
print(f"  Duplicates: {qqq_df['CNTSTUID'].duplicated().sum()}")

# Missing rate per context variable
print("\nMissing rate per contextual variable:")
for col in qqq_context_cols:
    nan_pct = 100 * qqq_df[col].isna().mean()
    print(f"  {col:12s}: {nan_pct:5.1f}%")

# Descriptive stats on plausible values
print("\nPlausible value statistics (math achievement, 500=OECD mean):")
print(qqq_df[qqq_pv_cols].describe().round(1))

Reading QQQ file with column filtering...
QQQ read in 8.8 seconds
  Shape: (613744, 21)
  Memory: 54.0 MB
  RAM available: 5.14 GB

QQQ health check:
  Unique students: 613,744
  Duplicates: 0

Missing rate per contextual variable:
  ESCS        :   4.1%
  ST004D01T   :   0.0%
  ST019AQ01T  :   4.2%
  MISCED      :   5.4%
  FISCED      :   7.4%
  HOMEPOS     :   2.6%
  ICTHOME     :  46.0%
  MATHEFF     :  23.5%
  ANXMAT      :  22.5%
  BELONG      :   8.5%

Plausible value statistics (math achievement, 500=OECD mean):
        PV1MATH   PV2MATH   PV3MATH   PV4MATH   PV5MATH   PV6MATH   PV7MATH   PV8MATH   PV9MATH  PV10MATH
count  613744.0  613744.0  613744.0  613744.0  613744.0  613744.0  613744.0  613744.0  613744.0  613744.0
mean      440.9     441.0     441.0     440.9     440.9     440.8     440.9     440.8     440.8     440.9
std       101.8     101.8     101.8     101.9     101.9     101.8     101.8     101.8     101.8     101.8
min         0.0      45.9      55.5       0.0      

In [ ]:
# Read TIM file - only EFFORTT needed as effort/motivation indicator

# TIM is 633 MB with 172 columns but we only need 2 columns (CNTSTUID + EFFORTT),
# so this reads very quickly.

print("Reading TIM file with column filtering...")
start = time.time()

tim_df, _ = pyreadstat.read_sav(
    str(TIM_PATH),
    usecols=tim_cols_to_read,
)

# Downcast EFFORTT to float32
if pd.api.types.is_numeric_dtype(tim_df["EFFORTT"]):
    tim_df["EFFORTT"] = tim_df["EFFORTT"].astype("float32")

elapsed = time.time() - start
print(f"TIM read in {elapsed:.1f} seconds")
print(f"  Shape: {tim_df.shape}")
print(f"  Memory: {tim_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print("\nTIM health check:")
print(f"  Unique students: {tim_df['CNTSTUID'].nunique():,}")
print(f"  Duplicates: {tim_df['CNTSTUID'].duplicated().sum()}")
print(f"  EFFORTT missing: {100 * tim_df['EFFORTT'].isna().mean():.1f}%")

print("\nEFFORTT statistics (effort/motivation indicator):")
print(tim_df["EFFORTT"].describe().round(2))

Reading TIM file with column filtering...
TIM read in 1.0 seconds
  Shape: (613744, 2)
  Memory: 7.4 MB

TIM health check:
  Unique students: 613,744
  Duplicates: 0
  EFFORTT missing: 4.7%

EFFORTT statistics (effort/motivation indicator):
count      585143.00
mean        59136.25
std        168606.38
min            68.00
25%         31581.00
50%         47341.00
75%         65563.00
max      77394320.00
Name: EFFORTT, dtype: float64


In [ ]:
# Merge COG, QQQ, and TIM on CNTSTUID

# Strategy: Start with an inner join to ensure all students have data from
# all three sources. We report merge statistics to verify no unexpected loss.
# If significant rows are dropped, we'll investigate and potentially switch
# to a left join.

print("Starting merge process...")
print(f"  COG rows before merge: {len(cog_df):,}")
print(f"  QQQ rows before merge: {len(qqq_df):,}")
print(f"  TIM rows before merge: {len(tim_df):,}")

# Check ID overlap before merging
cog_ids = set(cog_df["CNTSTUID"])
qqq_ids = set(qqq_df["CNTSTUID"])
tim_ids = set(tim_df["CNTSTUID"])

print(f"\nID overlap analysis:")
print(f"  Unique IDs in COG: {len(cog_ids):,}")
print(f"  Unique IDs in QQQ: {len(qqq_ids):,}")
print(f"  Unique IDs in TIM: {len(tim_ids):,}")
print(f"  In all three files: {len(cog_ids & qqq_ids & tim_ids):,}")
print(f"  In COG but not QQQ: {len(cog_ids - qqq_ids):,}")
print(f"  In COG but not TIM: {len(cog_ids - tim_ids):,}")
print(f"  In QQQ but not COG: {len(qqq_ids - cog_ids):,}")

# Free set memory
del cog_ids, qqq_ids, tim_ids
gc.collect()

print("\nMerging COG + QQQ on CNTSTUID (inner join)...")
start = time.time()

# Step 1: Merge COG with QQQ (inner join)
merged_df = cog_df.merge(qqq_df, on="CNTSTUID", how="inner", validate="one_to_one")
print(f"  After COG+QQQ merge: {merged_df.shape} ({time.time()-start:.1f}s)")

# Free QQQ memory - no longer needed
del qqq_df
gc.collect()

# Step 2: Merge with TIM (left join so we keep students without EFFORTT)
print("\nMerging with TIM on CNTSTUID (left join to preserve all students)...")
start = time.time()
merged_df = merged_df.merge(tim_df, on="CNTSTUID", how="left", validate="one_to_one")
print(f"  After +TIM merge: {merged_df.shape} ({time.time()-start:.1f}s)")

# Free TIM memory
del tim_df
gc.collect()

# Also free the original COG DataFrame since merged_df now holds everything
del cog_df
gc.collect()

print("\nFinal merged DataFrame:")
print(f"  Shape: {merged_df.shape}")
print(f"  Memory: {merged_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"  RAM available: {psutil.virtual_memory().available / 1e9:.2f} GB")

Starting merge process...
  COG rows before merge: 613,744
  QQQ rows before merge: 613,744
  TIM rows before merge: 613,744

ID overlap analysis:
  Unique IDs in COG: 613,744
  Unique IDs in QQQ: 613,744
  Unique IDs in TIM: 613,744
  In all three files: 613,744
  In COG but not QQQ: 0
  In COG but not TIM: 0
  In QQQ but not COG: 0

Merging COG + QQQ on CNTSTUID (inner join)...
  After COG+QQQ merge: (613744, 1193) (0.7s)

Merging with TIM on CNTSTUID (left join to preserve all students)...
  After +TIM merge: (613744, 1194) (0.1s)

Final merged DataFrame:
  Shape: (613744, 1194)
  Memory: 2.94 GB
  RAM available: 5.28 GB


In [ ]:
# Create math_score target and retain all 10 plausible values

# Hybrid strategy (decision made in `01_data_merging.ipynb` planning):
#   - `03_ml_model.ipynb` will fit 10 separate models (one per PV) and pool results
#     via Rubin's Rules. This is the methodologically correct approach for
#     plausible values because it propagates measurement uncertainty from
#     the IRT ability estimation into the ML analysis.
#   - math_score (mean of PVs) is retained for sensitivity analysis and
#     rapid exploratory modeling during development.
#
# By keeping both the aggregated math_score AND the 10 individual PVs, we
# preserve maximum flexibility for downstream analysis without committing
# to either approach at this stage.

pv_cols = [f"PV{i}MATH" for i in range(1, 11)]

# Create the aggregated target for sensitivity analysis
merged_df["math_score"] = merged_df[pv_cols].mean(axis=1).astype("float32")

# Report descriptive statistics on the aggregated target
print("Aggregated target variable (math_score) statistics:")
print(merged_df["math_score"].describe().round(2))

# Verify correlations between math_score and individual PVs
# Expected: very high (>0.99) since math_score is the mean
print("\nCorrelation of math_score with each PV (expected >0.99):")
for pv in pv_cols:
    corr = merged_df[["math_score", pv]].corr().iloc[0, 1]
    print(f"  {pv:10s}: {corr:.4f}")

# Check for any NaN in math_score
nan_count = merged_df["math_score"].isna().sum()
print(f"\nNaN count in math_score: {nan_count}")

# Descriptive statistics on individual PVs
# This is the data that will drive the Rubin's Rules analysis in `03_ml_model.ipynb`, so we want to confirm they look reasonable before proceeding.
print("\nIndividual plausible values statistics:")
pv_stats = merged_df[pv_cols].describe().round(2)
print(pv_stats)

# Check between-PV variance at student level (a diagnostic for PV reliability)
# This gives us an early sense of measurement uncertainty in the dataset
pv_within_student_std = merged_df[pv_cols].std(axis=1)
print(f"\nWithin-student PV standard deviation (measurement uncertainty indicator):")
print(f"  Mean : {pv_within_student_std.mean():.2f}")
print(f"  Median: {pv_within_student_std.median():.2f}")
print(f"  Max  : {pv_within_student_std.max():.2f}")
print(f"  (Typical PISA values: mean SD around 5-15 points on the 500-scale)")

print(f"\nDataFrame shape (PVs retained): {merged_df.shape}")
print(f"Memory: {merged_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

gc.collect()

Aggregated target variable (math_score) statistics:
count    613744.00
mean        440.90
std          98.44
min          39.17
25%         364.75
50%         432.39
75%         509.79
max         843.29
Name: math_score, dtype: float64

Correlation of math_score with each PV (expected >0.99):
  PV1MATH   : 0.9668
  PV2MATH   : 0.9666
  PV3MATH   : 0.9668
  PV4MATH   : 0.9667
  PV5MATH   : 0.9668
  PV6MATH   : 0.9666
  PV7MATH   : 0.9667
  PV8MATH   : 0.9668
  PV9MATH   : 0.9667
  PV10MATH  : 0.9666

NaN count in math_score: 0

Individual plausible values statistics:
         PV1MATH    PV2MATH    PV3MATH    PV4MATH    PV5MATH    PV6MATH    PV7MATH    PV8MATH    PV9MATH   PV10MATH
count  613744.00  613744.00  613744.00  613744.00  613744.00  613744.00  613744.00  613744.00  613744.00  613744.00
mean      440.87     440.97     441.02     440.95     440.95     440.80     440.88     440.85     440.81     440.90
std       101.84     101.77     101.85     101.87     101.89     101.78     10

0

In [ ]:
# Missing data analysis across three variable categories

# Different variable types have different missing mechanisms:
#   - math_score and PVs: complete by design (IRT estimation fills all)
#   - Contextual variables: MAR - student skipped items or module not administered
#   - Process data (CM columns): MCAR by matrix-sampled design - most cells
#     are missing because the student did not see that specific item

print("=" * 70)
print("MISSING DATA ANALYSIS")
print("=" * 70)

# Category 1: Target variables (math_score and all PVs)
print("\n1. TARGET VARIABLES")
print("-" * 70)
target_cols = ["math_score"] + pv_cols
for col in target_cols:
    miss_count = merged_df[col].isna().sum()
    miss_pct = 100 * miss_count / len(merged_df)
    print(f"  {col:12s}: {miss_count} missing ({miss_pct:.2f}%)")

# Category 2: Contextual variables (from QQQ and TIM)
print("\n2. CONTEXTUAL VARIABLES")
print("-" * 70)
context_vars = ["ESCS", "ST004D01T", "ST019AQ01T", "MISCED", "FISCED",
                "HOMEPOS", "ICTHOME", "MATHEFF", "ANXMAT", "BELONG", "EFFORTT"]
context_missing = merged_df[context_vars].isna().sum().sort_values(ascending=False)
context_missing_pct = (100 * context_missing / len(merged_df)).round(2)

missing_summary = pd.DataFrame({
    "missing_count": context_missing,
    "missing_pct": context_missing_pct
})
print(missing_summary)

# Category 3: Process data (CM columns)
print("\n3. PROCESS DATA (CM columns - matrix-sampled design)")
print("-" * 70)
process_cols_all = [c for c in merged_df.columns 
                    if c.startswith("CM") and c not in ("CNTSTUID", "CNTSCHID", "CNT")]

total_process_cells = len(merged_df) * len(process_cols_all)
missing_process_cells = merged_df[process_cols_all].isna().sum().sum()
overall_process_missing = 100 * missing_process_cells / total_process_cells
print(f"  Total process columns: {len(process_cols_all)}")
print(f"  Overall missing rate: {overall_process_missing:.1f}%")
print(f"  (High by design: matrix-sampled booklets)")

# Per-suffix missing rates
print("\n  Missing rate by process feature type:")
for suffix in ["TT", "A", "V", "F", "VS"]:
    suffix_cols = [c for c in process_cols_all if c.endswith(suffix)]
    if suffix_cols:
        suffix_missing_pct = 100 * merged_df[suffix_cols].isna().mean().mean()
        print(f"    {suffix:4s}: {suffix_missing_pct:.1f}% (across {len(suffix_cols)} columns)")

# Per-student: how many items each student has data for
print("\n  Items seen per student (based on TT columns):")
tt_cols = [c for c in process_cols_all if c.endswith("TT")]
items_per_student = merged_df[tt_cols].notna().sum(axis=1)
print(f"    Min : {items_per_student.min()}")
print(f"    25% : {items_per_student.quantile(0.25):.0f}")
print(f"    50% : {items_per_student.median():.0f}")
print(f"    75% : {items_per_student.quantile(0.75):.0f}")
print(f"    Max : {items_per_student.max()}")
print(f"    Mean: {items_per_student.mean():.1f}")

zero_process_students = (items_per_student == 0).sum()
print(f"\n  Students with zero process data: {zero_process_students:,} "
      f"({100*zero_process_students/len(merged_df):.1f}%)")

MISSING DATA ANALYSIS

1. TARGET VARIABLES
----------------------------------------------------------------------
  math_score  : 0 missing (0.00%)
  PV1MATH     : 0 missing (0.00%)
  PV2MATH     : 0 missing (0.00%)
  PV3MATH     : 0 missing (0.00%)
  PV4MATH     : 0 missing (0.00%)
  PV5MATH     : 0 missing (0.00%)
  PV6MATH     : 0 missing (0.00%)
  PV7MATH     : 0 missing (0.00%)
  PV8MATH     : 0 missing (0.00%)
  PV9MATH     : 0 missing (0.00%)
  PV10MATH    : 0 missing (0.00%)

2. CONTEXTUAL VARIABLES
----------------------------------------------------------------------
            missing_count  missing_pct
ICTHOME            282163        45.97
MATHEFF            144231        23.50
ANXMAT             137952        22.48
BELONG              52405         8.54
FISCED              45428         7.40
MISCED              33442         5.45
EFFORTT             28601         4.66
ST019AQ01T          25914         4.22
ESCS                25468         4.15
HOMEPOS             15730 

In [ ]:
# Final cleanup and metadata preparation before saving

# This cell does three things:
#   1. Verifies the final column structure and order
#   2. Adds helpful metadata columns (items_seen) for downstream filtering
#   3. Logs a full summary of the dataset characteristics

# Add a helper column: number of math items each student saw
# This lets us filter out students with zero process data in `02_feature_engineering.ipynb`
# without re-computing this expensive operation
tt_cols = [c for c in merged_df.columns if c.startswith("CM") and c.endswith("TT")]
merged_df["n_items_seen"] = merged_df[tt_cols].notna().sum(axis=1).astype("int16")

# Organize columns into logical groups for clarity
id_group = ["CNTSTUID", "CNT", "CNTSCHID"]
target_group = ["math_score"] + [f"PV{i}MATH" for i in range(1, 11)]
context_group = ["ESCS", "ST004D01T", "ST019AQ01T", "MISCED", "FISCED",
                 "HOMEPOS", "ICTHOME", "MATHEFF", "ANXMAT", "BELONG", "EFFORTT"]
meta_group = ["n_items_seen"]
process_group = [c for c in merged_df.columns if c.startswith("CM")]

# Reorder columns: IDs, targets, contextual, metadata, then process data
ordered_cols = id_group + target_group + context_group + meta_group + process_group

# Verify all columns are accounted for
all_accounted = set(ordered_cols) == set(merged_df.columns)
print(f"All columns accounted for in grouping: {all_accounted}")
if not all_accounted:
    missing_from_order = set(merged_df.columns) - set(ordered_cols)
    extra_in_order = set(ordered_cols) - set(merged_df.columns)
    print(f"  Missing from order: {missing_from_order}")
    print(f"  Extra in order: {extra_in_order}")

merged_df = merged_df[ordered_cols]

# Final dataset summary
print("\nFINAL DATASET SUMMARY")
print("-" * 70)
print(f"Total rows       : {len(merged_df):,}")
print(f"Total columns    : {len(merged_df.columns)}")
print(f"  ID columns     : {len(id_group)}")
print(f"  Target columns : {len(target_group)} (math_score + 10 PVs)")
print(f"  Context vars   : {len(context_group)}")
print(f"  Metadata       : {len(meta_group)}")
print(f"  Process columns: {len(process_group)}")
print(f"\nMemory           : {merged_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"RAM available    : {psutil.virtual_memory().available / 1e9:.2f} GB")

# Dtype summary
print(f"\nDtype distribution:")
print(merged_df.dtypes.value_counts())

# Preview first and last few columns
print(f"\nFirst 5 column names: {list(merged_df.columns[:5])}")
print(f"Last 5 column names : {list(merged_df.columns[-5:])}")

All columns accounted for in grouping: True

FINAL DATASET SUMMARY
----------------------------------------------------------------------
Total rows       : 613,744
Total columns    : 1196
  ID columns     : 3
  Target columns : 11 (math_score + 10 PVs)
  Context vars   : 11
  Metadata       : 1
  Process columns: 1170

Memory           : 2.94 GB
RAM available    : 5.63 GB

Dtype distribution:
float32    1192
float64       2
str           1
int16         1
Name: count, dtype: int64

First 5 column names: ['CNTSTUID', 'CNT', 'CNTSCHID', 'math_score', 'PV1MATH']
Last 5 column names : ['CMA135Q04TT', 'CMA135Q04F', 'CMA135Q04A', 'CMA135Q04V', 'CMA135Q04VS']


In [ ]:
# Save the merged dataset to parquet format

# Parquet advantages over SAV/CSV:
#   - Columnar storage: fast reads when only subset of columns needed
#   - Compression: file size smaller than uncompressed equivalents
#   - Preserves dtypes: float32 stays float32 (no re-conversion)
#   - Fast to read: seconds vs minutes for SAV

# All 613,744 students are saved. Filtering of students without process
# data (n_items_seen == 0) will occur in Notebook 02, preserving the full
# merged dataset for methodological transparency.

print(f"Saving to: {OUTPUT_PATH}")
print(f"DataFrame shape: {merged_df.shape}")
print(f"In-memory size : {merged_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

start = time.time()

merged_df.to_parquet(
    OUTPUT_PATH,
    engine="pyarrow",
    compression="snappy",
    index=False,
)

elapsed = time.time() - start
file_size_gb = OUTPUT_PATH.stat().st_size / 1e9

print(f"\nSaved in {elapsed:.1f} seconds")
print(f"File size: {file_size_gb:.2f} GB")
print(f"Compression ratio: "
      f"{merged_df.memory_usage(deep=True).sum() / OUTPUT_PATH.stat().st_size:.2f}x")

# Verify the file can be read back correctly
print("\nVerifying parquet file integrity...")
start = time.time()
verify_df = pd.read_parquet(OUTPUT_PATH, columns=["CNTSTUID", "math_score", "ESCS", "n_items_seen"])
elapsed = time.time() - start

print(f"  Read back in {elapsed:.2f} seconds (partial read, 4 columns)")
print(f"  Verification shape: {verify_df.shape}")
print(f"  math_score mean matches: "
      f"{np.isclose(verify_df['math_score'].mean(), merged_df['math_score'].mean())}")
print(f"  Students with process data (n_items_seen > 0): "
      f"{(verify_df['n_items_seen'] > 0).sum():,}")
print(f"  Students without process data: "
      f"{(verify_df['n_items_seen'] == 0).sum():,}")

del verify_df
gc.collect()

Saving to: /Users/mrved/Desktop/PISA-Process-Data-Analysis/data/analysis/pisa2022_ml_dataset.parquet
DataFrame shape: (613744, 1196)
In-memory size : 2.94 GB

Saved in 10.7 seconds
File size: 0.31 GB
Compression ratio: 9.43x

Verifying parquet file integrity...
  Read back in 0.24 seconds (partial read, 4 columns)
  Verification shape: (613744, 4)
  math_score mean matches: True
  Students with process data (n_items_seen > 0): 555,784
  Students without process data: 57,960


0

In [ ]:
# Final verification and memory cleanup for `01_data_merging.ipynb`

# Clean up the large in-memory DataFrame now that it has been saved to disk.
# This frees RAM before moving to the `02_feature_engineering.ipynb` notebook.

print("`01_data_merging.ipynb` completed successfully.")
print("-" * 70)
print(f"Output file: {OUTPUT_PATH}")
print(f"File size  : {OUTPUT_PATH.stat().st_size / 1e9:.2f} GB")
print(f"Total rows : {len(merged_df):,}")
print(f"Total cols : {len(merged_df.columns)}")

print("\nKey dataset characteristics:")
print(f"  Students with process data   : 555,784 (90.6%)")
print(f"  Students without process data: 57,960  (9.4%, will be filtered in NB02)")
print(f"  Target: math_score + 10 plausible values for Rubin's Rules analysis")
print(f"  Process features: 234 items x 5 behavioral dimensions = 1170 columns")

print("\nNext steps - `02_feature_engineering.ipynb`:")
print("  1. Load parquet file (fast, a few seconds)")
print("  2. Filter students where n_items_seen == 0")
print("  3. Compute behavioral indicators per student:")
print("       - rapid_guess_ratio: share of items answered suspiciously fast")
print("       - time_variability: within-student std of response times")
print("       - effort_regulation: trend in response time across items")
print("       - total_actions, total_visits, visit_patterns")
print("  4. Handle outliers in timing columns")
print("  5. Log-transform heavily skewed variables")
print("  6. Save feature matrix as pisa2022_features.parquet")

print("\nNext steps - `03_ml_model.ipynb`:")
print("  1. Fit 10 models (one per PV) for main Rubin's Rules analysis")
print("  2. Fit 1 sensitivity model on aggregated math_score")
print("  3. Hyperparameter tuning on PV1, applied consistently across all 10")

print("\nNext steps - `04_shap_xai.ipynb`:")
print("  1. Pool SHAP values across 10 PV models with Rubin's Rules")
print("  2. Generate interpretability plots (summary, dependence, force)")
print("  3. Interpret findings through Dual-Process Theory lens")

# Release main DataFrame memory
del merged_df
gc.collect()

print(f"\nRAM available after cleanup: "
      f"{psutil.virtual_memory().available / 1e9:.2f} GB")
print("\nReady to proceed to Notebook 02.")

Notebook 01 completed successfully.
----------------------------------------------------------------------
Output file: /Users/mrved/Desktop/PISA-Process-Data-Analysis/data/analysis/pisa2022_ml_dataset.parquet
File size  : 0.31 GB
Total rows : 613,744
Total cols : 1196

Key dataset characteristics:
  Students with process data   : 555,784 (90.6%)
  Students without process data: 57,960  (9.4%, will be filtered in NB02)
  Target: math_score + 10 plausible values for Rubin's Rules analysis
  Process features: 234 items x 5 behavioral dimensions = 1170 columns

Next steps - Notebook 02 (feature engineering):
  1. Load parquet file (fast, a few seconds)
  2. Filter students where n_items_seen == 0
  3. Compute behavioral indicators per student:
       - rapid_guess_ratio: share of items answered suspiciously fast
       - time_variability: within-student std of response times
       - effort_regulation: trend in response time across items
       - total_actions, total_visits, visit_pattern

# Technical Report
## Data Integration for PISA 2022 Process Data Analysis

**Project:** Analyzing the effects of time-based behavioral patterns on mathematics achievement using explainable machine learning with Dual-Process Theory interpretation
**Notebook:** `01_veri_birlestirme.ipynb`
**Date:** April 2026
**Environment:** Python 3.11.15 / macOS (Apple Silicon) / Conda `pisa`

---

## 1. Overview

This notebook merges three PISA 2022 international student data files into a single analysis-ready dataset. The workflow extracts mathematics cognitive process data, plausible value outcome measures, contextual background variables, and an effort/motivation indicator from the publicly released PISA 2022 data (OECD, 2024). The output is a compressed Parquet file containing 613,744 students and 1,196 variables, serving as the foundation for subsequent feature engineering, machine learning modeling, and SHAP-based interpretability analyses.

---

## 2. Data Sources

Three SPSS files released by the OECD on the PISA Data Repository were used:

| File | Size | Rows | Columns | Role in Study |
|------|------|------|---------|---------------|
| `CY08MSP_STU_COG.SAV` | 3.73 GB | 613,744 | ~5,023 | Item-level cognitive responses and process data (actions, visits, timing) |
| `CY08MSP_STU_QQQ.SAV` | 2.10 GB | 613,744 | 1,278 | Student questionnaire and plausible value achievement estimates |
| `CY08MSP_STU_TIM.SAV` | 0.66 GB | 613,744 | 172 | Timing data from the background questionnaire; only the total effort indicator was retained |

All three files shared the same 613,744 unique student identifiers (`CNTSTUID`) across 80 participating countries and economies, confirming full overlap without loss during integration.

---

## 3. Variable Selection

### 3.1 Outcome Measures
Ten plausible values for mathematics achievement (`PV1MATH`–`PV10MATH`) were extracted from the QQQ file. In PISA, plausible values are draws from each student's posterior ability distribution estimated via a population Item Response Theory (IRT) model (von Davier et al., 2009). All ten values were retained to support a Rubin's Rules aggregation in the downstream modeling notebooks (Rubin, 1987; Little & Rubin, 2019), with their arithmetic mean (`math_score`) additionally computed as a secondary target for sensitivity analysis.

### 3.2 Process Features
PISA 2022 encodes mathematics item-level process data in columns prefixed with `CM` (Cognitive Mathematics), distinguished by suffix:

| Suffix | Meaning | Columns |
|--------|---------|---------|
| `TT` | Total time spent on the item (ms) | 234 |
| `A` | Number of actions performed | 234 |
| `V` | Number of visits to the item | 234 |
| `F` | First-visit duration | 234 |
| `VS` | Visit sequence encoding | 234 |

All 1,170 process columns across 234 mathematics items were retained. The 199 `CM*S` score columns were excluded at this stage because this study focuses on behavioral process patterns rather than item-level cognitive correctness; they can be reintroduced in a subsequent analysis if needed. Other prefix groups in the COG file (`DM*`, `PM*`, reading, and science items) were excluded as outside the scope of this study.

### 3.3 Contextual Variables
Ten contextual variables drawn from theoretical and empirical work in educational data mining and ILSA research were extracted from the QQQ file:

| Variable | Description | Theoretical Role |
|----------|-------------|------------------|
| `ESCS` | Economic, social and cultural status (composite index) | Socioeconomic control |
| `ST004D01T` | Gender | Demographic control |
| `ST019AQ01T` | Language spoken at home | Cultural/linguistic context |
| `MISCED` | Mother's education (ISCED) | Family educational capital |
| `FISCED` | Father's education (ISCED) | Family educational capital |
| `HOMEPOS` | Home possessions index | Material resources (substituted for HEDRES — see §4.2) |
| `ICTHOME` | ICT resources at home | Digital access |
| `MATHEFF` | Mathematics self-efficacy | Affective-motivational |
| `ANXMAT` | Mathematics anxiety | Affective-motivational |
| `BELONG` | Sense of belonging at school | Social-contextual |

### 3.4 Effort Indicator
`EFFORTT` was extracted from the TIM file as an indicator of total self-reported test-taking effort, measured in milliseconds. This variable operationalizes the effort regulation construct aligned with Self-Regulated Learning frameworks (Zimmerman, 2000) and connects to rapid-guessing research on low-stakes assessments (Wise & Kong, 2005).

---

## 4. Methodological Decisions

### 4.1 Chunked Reading of the COG File
The 3.73 GB COG file was read in 13 chunks of 50,000 rows using `pyreadstat`'s `read_file_in_chunks` generator, with column filtering applied at read time via the `usecols` parameter. This strategy was necessary because loading the full file naïvely would exceed available memory on typical research workstations. Total read time was 136.2 seconds (~2.3 minutes). After reading, all numeric columns were downcast from `float64` to `float32`, reducing in-memory footprint from 5.76 GB to 2.89 GB without loss of precision for count and timing data.

### 4.2 HEDRES Unavailable — HOMEPOS Substitution
The original analysis plan included `HEDRES` (Home Educational Resources). This index is not published in the PISA 2022 data release. `HOMEPOS` (home possessions index) was substituted as the broader parent index in the PISA indicator hierarchy. `HOMEPOS` is a component of the ESCS composite and captures a wider set of home resources, including educational items. This substitution should be disclosed in the methods section as a deviation from prior PISA cycles.

### 4.3 Plausible Value Handling — Hybrid Strategy
Two approaches to handling plausible values in machine learning analyses are recognized in the literature:

1. **Rubin's Rules aggregation:** Fit separate models for each of the ten plausible values and pool point estimates and variances across models. This propagates IRT measurement uncertainty into downstream inference (Rubin, 1987; Laukaityte & Wiberg, 2018).
2. **Mean aggregation:** Average the ten plausible values into a single target and fit one model. This simplifies computation but discards measurement uncertainty (Gabriel et al., 2017).

A hybrid strategy was adopted: all ten plausible values **and** their arithmetic mean (`math_score`) are stored in the output dataset. Rubin's Rules will serve as the primary analysis in downstream notebooks, while the mean-aggregated target will be used for sensitivity analyses and rapid exploratory modeling. This preserves methodological rigor while enabling flexible analysis paths.

A diagnostic correlation between `math_score` and individual plausible values was 0.9666–0.9668, indicating non-trivial within-student measurement variance and justifying the Rubin's Rules approach.

### 4.4 Join Strategy
COG and QQQ were joined using an inner merge on `CNTSTUID` with `validate="one_to_one"` to guarantee no duplicate identifiers across the join. TIM was then left-joined onto the result to preserve students lacking the effort indicator (4.7% missing). All three files shared identical student populations, so no rows were dropped. Final merged shape: **(613,744 × 1,194)** prior to target engineering.

### 4.5 Metadata Column — `n_items_seen`
A helper variable `n_items_seen` was computed as the number of mathematics items for which each student has non-missing `TT` values. This variable supports downstream filtering in Notebook 02 without recomputing expensive aggregations on the full process-column space.

### 4.6 No Filtering at Integration Stage
All 613,744 students were retained in the integration output. Filtering decisions — including removal of students without computer-based process data — were deferred to the feature engineering notebook. This preserves the raw merged dataset for methodological transparency and enables sensitivity tests with alternative filtering criteria.

---

## 5. Missing Data Analysis

### 5.1 Outcome Measures
All ten plausible values and the derived `math_score` are fully complete (0.00% missing), as expected given PISA's IRT-based plausible value generation.

### 5.2 Contextual Variables

| Variable | Missing % | Interpretation |
|----------|-----------|----------------|
| `ST004D01T` (gender) | 0.0% | Essentially complete |
| `HOMEPOS` | 2.6% | Low; item-nonresponse |
| `ESCS` | 4.1% | Low; driven by parental education missingness |
| `ST019AQ01T` | 4.2% | Low; item-nonresponse |
| `EFFORTT` | 4.7% | Low; module-dependent |
| `MISCED` | 5.4% | Low–moderate |
| `FISCED` | 7.4% | Low–moderate |
| `BELONG` | 8.5% | Moderate; rotated module |
| `ANXMAT` | 22.5% | High; administered to a rotated subsample |
| `MATHEFF` | 23.5% | High; administered to a rotated subsample |
| `ICTHOME` | 46.0% | Very high; optional module in many countries |

The elevated missingness on `MATHEFF`, `ANXMAT`, and `ICTHOME` is attributable to PISA's rotated questionnaire design and the optional nature of the ICT module. Missing patterns are expected to be Missing at Random (MAR) conditional on country and booklet assignment. Appropriate handling in Notebook 02 will involve either country-conditional imputation or inclusion of missingness indicators as features.

### 5.3 Process Data
The 1,170 process columns exhibit an overall missing rate of approximately 89%, which is a **design feature rather than a data quality concern**. PISA's matrix-sampled booklet design ensures that each student sees only a subset of items. The distribution of items seen per student is:

| Statistic | Items Seen (of 234) |
|-----------|--------------------|
| Mean | 26.1 |
| Median | ~26 |
| Max | 60 |
| Min | 0 |

Among the 613,744 students, 57,960 (9.4%) have zero process data. These students were administered the paper-based test variant or did not complete the computer-based assessment. They will be excluded in Notebook 02 because behavioral indicators cannot be computed without process data. Reporting this filtering decision transparently in the study's data-flow diagram is recommended.

---

## 6. Output

The merged dataset was written to `data/analysis/pisa2022_ml_dataset.parquet` using Snappy compression with the PyArrow engine.

| Property | Value |
|----------|-------|
| Rows | 613,744 |
| Columns | 1,196 |
| In-memory size (float32) | 2.94 GB |
| On-disk size (Parquet/Snappy) | 0.31 GB |
| Compression ratio | 9.43× |
| Full-file write time | 10.7 s |
| Partial-read time (4 columns) | 0.24 s |

The high compression ratio reflects the sparsity of the process-data matrix introduced by PISA's booklet design, which Parquet encodes efficiently. Partial column reads are several orders of magnitude faster than the original SPSS file, enabling rapid iteration in the downstream notebooks.

### 6.1 Column Organization
Columns are stored in the following order to facilitate downstream use:

1. **IDs** (3): `CNTSTUID`, `CNT`, `CNTSCHID`
2. **Targets** (11): `math_score`, `PV1MATH`–`PV10MATH`
3. **Contextual** (11): `ESCS`, `ST004D01T`, `ST019AQ01T`, `MISCED`, `FISCED`, `HOMEPOS`, `ICTHOME`, `MATHEFF`, `ANXMAT`, `BELONG`, `EFFORTT`
4. **Metadata** (1): `n_items_seen`
5. **Process** (1,170): `CM*TT`, `CM*A`, `CM*V`, `CM*F`, `CM*VS` across 234 items

---

## 7. Data Flow Summary

```
PISA 2022 SAV files (6.49 GB total)
              |
              v
  +----------+----------+----------+
  |          |          |          |
 COG        QQQ        TIM         |
 3.73 GB    2.10 GB    0.66 GB     |
 chunked    filtered   filtered    |
 read       read       read        |
  |          |          |          |
  +----------v----------+          |
           merge                   |
      (one-to-one on               |
        CNTSTUID)                  |
             |                     |
             v                     |
    target engineering             |
  (math_score = mean(PV1..PV10))   |
             |                     |
             v                     |
   float32 downcast +              |
   column reordering               |
             |                     |
             v                     |
   pisa2022_ml_dataset.parquet     |
   (613,744 rows x 1,196 cols)     |
   0.31 GB on disk                 |
```

---

## 8. Next Steps

The output file feeds into three downstream notebooks:

- **`02_feature_engineering.ipynb`:** filter students with `n_items_seen == 0`, handle outliers in timing variables, apply log transformations to skewed distributions, and compute behavioral indicators including rapid-guess ratio, time variability, effort regulation, and action/visit aggregates.
- **`03_ml_model.ipynb`:** fit ten gradient-boosted models (one per plausible value) for the primary Rubin's Rules analysis, plus one sensitivity model on `math_score`.
- **`04_shap_xai.ipynb`:** pool SHAP values across the ten plausible value models using Rubin's Rules and interpret feature contributions through the Dual-Process Theory framework (Evans & Stanovich, 2013; Kahneman, 2011).